# 金融/股票常见指标教程：从“为什么需要”到“如何使用”

> 面向初学者。示例股票：`603993`（洛阳钼业）。  
> 数据源：优先使用 AKShare 的真实数据；接口不可用时，才使用明确标记的演示数据。  
> 绘图：全部使用 Matplotlib，并把计算公式直接写成 Python。

## 先建立一个直觉

技术指标没有创造新信息。它只是对 `价格、成交量、财务数据` 做压缩、平滑或比较，让某种市场状态更容易看见：

```text
价格是否在涨？        → 均线、MACD
涨得是否过快？        → ROC、RSI、KDJ
现在波动是否异常？    → 历史波动率、ATR、布林带
上涨有没有成交量配合？→ OBV、PVT、CMF、MFI
这只股票是否跟随大盘？→ Beta、相关性、Alpha
公司是否贵、质量如何？→ PE、PB、ROE、利润率
```

因此，学习一个指标时不要只背“金叉买、死叉卖”。每个指标都要问八个问题：

1. **为什么需要它？** 原始价格图哪里看不清？
2. **它怎么来的？** 它在压缩什么信息？
3. **公式做了什么？** 平均、差分、标准化还是累计？
4. **数值实际表示什么？** 单位是什么，0、1、50、100 各意味着什么？
5. **图应该怎么看？** 看水平、方向、交叉还是背离？
6. **怎么应用？** 适合趋势、震荡、风控还是评价？
7. **和谁配合？** 哪些指标提供互补信息？
8. **什么时候会失效？** 它的滞后、噪声和假信号在哪里？

本 Notebook 按这个模板逐项讲解，并把公式、代码、图形放在一起。

> 重要：指标描述历史，不保证未来。一个指标算得正确，只代表测量正确，不代表交易策略有效。

## 0. 环境与数据

项目使用 `uv`：

```bash
uv sync
uv run jupyter notebook
```

如果缺依赖：

```bash
uv add akshare pandas numpy matplotlib
```

In [ ]:
import importlib.util
for pkg in ["akshare", "pandas", "numpy", "matplotlib"]:
    print(f"{pkg:10s}", "OK" if importlib.util.find_spec(pkg) else "MISSING")

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (14, 6)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

## 1. 下载 603993 日线数据

我们需要的基础行情字段：

| 字段 | 含义 |
|---|---|
| `Open` | 开盘价 |
| `High` | 最高价 |
| `Low` | 最低价 |
| `Close` | 收盘价 |
| `Volume` | 成交量 |
| `Amount` | 成交额 |
| `Turnover` | 换手率，若接口返回 |

默认使用前复权 `qfq`，因为图形更连续，适合学习技术指标。

In [ ]:
def standardize_akshare_stock(raw):
    rename = {
        "日期": "Date",
        "开盘": "Open",
        "最高": "High",
        "最低": "Low",
        "收盘": "Close",
        "成交量": "Volume",
        "成交额": "Amount",
        "振幅": "Amplitude",
        "涨跌幅": "PctChange",
        "涨跌额": "Change",
        "换手率": "Turnover",
    }
    df = raw.rename(columns=rename).copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.set_index("Date").sort_index()
    keep = ["Open", "High", "Low", "Close", "Volume", "Amount", "Amplitude", "PctChange", "Change", "Turnover"]
    keep = [c for c in keep if c in df.columns]
    for c in keep:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df[keep].dropna(subset=["Open", "High", "Low", "Close"])


def make_demo_ohlcv(seed=603993, start="2020-01-01", periods=1100):
    rng = np.random.default_rng(seed)
    dates = pd.bdate_range(start, periods=periods)
    ret = rng.normal(0.0004, 0.022, size=periods)
    close = 5 * np.exp(np.cumsum(ret))
    open_ = close * (1 + rng.normal(0, 0.006, size=periods))
    high = np.maximum(open_, close) * (1 + rng.random(periods) * 0.018)
    low = np.minimum(open_, close) * (1 - rng.random(periods) * 0.018)
    volume = rng.integers(200_000, 20_000_000, size=periods)
    amount = volume * close
    turnover = rng.uniform(0.3, 6.0, size=periods)
    return pd.DataFrame({
        "Open": open_, "High": high, "Low": low, "Close": close,
        "Volume": volume, "Amount": amount,
        "Amplitude": (high - low) / close * 100,
        "PctChange": pd.Series(close).pct_change().fillna(0).values * 100,
        "Change": pd.Series(close).diff().fillna(0).values,
        "Turnover": turnover,
    }, index=dates)


def load_stock_603993(start="20200101", end="20260101", adjust="qfq", use_cache=True):
    cache = DATA_DIR / f"603993_daily_{start}_{end}_{adjust or 'none'}.csv"
    if use_cache and cache.exists():
        return pd.read_csv(cache, index_col="Date", parse_dates=True)
    try:
        import akshare as ak
        raw = ak.stock_zh_a_hist(symbol="603993", period="daily", start_date=start, end_date=end, adjust=adjust)
        df0 = standardize_akshare_stock(raw)
        df0.to_csv(cache, index_label="Date")
        print("AKShare 数据下载成功")
        return df0
    except Exception as e:
        print("AKShare 下载失败，使用演示数据。错误：", repr(e))
        return make_demo_ohlcv()


df = load_stock_603993()
df.tail()

In [ ]:
def plot_lines(data, columns, title, hlines=None, ylabel=None):
    fig, ax = plt.subplots(figsize=(14, 5))
    for col in columns:
        ax.plot(data.index, data[col], label=col)
    if hlines:
        for y, color, style, label in hlines:
            ax.axhline(y, color=color, linestyle=style, alpha=0.7, label=label)
    ax.set_title(title)
    ax.set_ylabel(ylabel or "")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()


def plot_price_with(data, columns, title):
    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    axes[0].plot(data.index, data["Close"], label="Close")
    axes[0].set_title("Price")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()
    for col in columns:
        axes[1].plot(data.index, data[col], label=col)
    axes[1].set_title(title)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_lines(df, ["Close"], "603993 Close Price")

## 2. 价格与收益类指标：所有分析的地基

价格告诉我们资产值多少钱，但不能直接回答“赚了多少”或“风险多大”。例如从 5 元涨到 6 元和从 50 元涨到 51 元都涨了 1 元，经济意义完全不同。因此金融分析通常先把价格转换为**收益率**，再计算累计收益、波动率、夏普、Beta、VaR 等指标。

这一章的依赖关系是：

```text
Close → 单期收益率 → 累计净值 → 回撤
                  ├→ 波动率 / 夏普 / VaR
                  └→ Beta / Alpha
```

### 2.1 简单收益率 Simple Return

**为什么需要它**  
直接比较价格差会受价格基数影响。收益率把变化换成比例，让不同价格、不同股票可以比较。

**它怎么来的**  
用今天价格除以昨天价格，再减 1：

```text
Return_t = Close_t / Close_{t-1} - 1
```

**实际含义**  
`0.03` 表示上涨 3%，`-0.03` 表示下跌 3%。它是有方向的无量纲比例。注意上涨 10% 后再下跌 10% 并不会回到原点：`1.1 × 0.9 = 0.99`。

**如何应用**

- 计算每日涨跌和累计净值；
- 比较不同股票的表现；
- 作为波动率、夏普、VaR、Beta 等统计指标的输入；
- 在策略回测里计算每天账户收益。

**图怎么看**  
围绕 0 上下波动。尖峰表示极端涨跌；波动密集放大表示市场风险上升。

**和其他指标的关系**  
它是后续风险与绩效指标的共同原料。技术指标多直接从价格计算，风险指标多从收益率计算。

**局限与误区**  
简单收益率不能跨期直接相加；多期收益必须复利相乘。价格复权方式也会影响收益率，长期股票研究不能忽略分红送转。

In [ ]:
df["ret_1d"] = df["Close"].pct_change()
plot_price_with(df, ["ret_1d"], "Simple Daily Return")
df[["Close", "ret_1d"]].tail()

### 2.2 对数收益率 Log Return

**为什么需要它**  
简单收益率跨期要连乘，不方便数学建模。对数把乘法变成加法。

**它怎么来的**

```text
LogReturn_t = ln(Close_t / Close_{t-1})
```

多期对数收益可以直接相加：

```text
ln(P2/P0) = ln(P1/P0) + ln(P2/P1)
```

**实际含义**  
在日涨跌很小时，它与简单收益率几乎相同。例如简单收益 1% 时，对数收益约 0.995%。

**如何应用**

- 时间序列和统计模型；
- 需要跨期相加的收益分析；
- 常见的收益分布研究。

**图怎么看**  
形状与简单日收益高度相似。极端涨跌时，两者差异更明显。

**和其他指标的关系**  
波动率既可基于简单收益，也可基于对数收益，但一份研究中必须统一口径。

**局限与误区**  
对数收益不是账户真实百分比收益。它要求价格为正；对普通股票通常成立，但不能机械套到允许非正值的变量。

In [ ]:
df["log_ret_1d"] = np.log(df["Close"] / df["Close"].shift(1))
plot_price_with(df, ["log_ret_1d"], "Log Daily Return")
df[["ret_1d", "log_ret_1d"]].tail()

### 2.3 累计收益 Cumulative Return

**为什么需要它**  
单日收益只描述一天，投资者更关心“从起点持有到现在一共赚了多少”。

**它怎么来的**  
把每天的财富增长因子连续相乘：

```text
CumReturn_t = ∏(1 + Return_i) - 1
```

**实际含义**  
累计收益 `0.50` 表示本金增长 50%；`-0.30` 表示亏损 30%。`1 + CumReturn` 就是从 1 元起步的净值曲线。

**如何应用**

- 画买入持有基准；
- 和策略净值比较；
- 计算年化收益与回撤；
- 比较不同区间的财富增长。

**图怎么看**  
重点看长期斜率、平台期和断崖式下跌。曲线终点只反映结果，路径决定投资者是否拿得住。

**和其他指标的关系**  
回撤来自累计净值相对历史高点的下降；年化收益来自累计收益按持有时间换算。

**局限与误区**  
累计收益高度依赖起止日期，不能只选最好看的区间。不同长度区间应配合年化收益比较。

In [ ]:
df["cum_ret"] = (1 + df["ret_1d"]).cumprod() - 1
plot_lines(df, ["cum_ret"], "Cumulative Return")
df[["cum_ret"]].tail()

### 2.4 回撤 Drawdown

**为什么需要它**  
波动率把上涨和下跌都视为风险，但投资者真正痛苦的是“从赚过的钱又亏回去”。回撤专门测量这种路径风险。

**它怎么来的**

```text
RunningPeak_t = max(Equity_0 ... Equity_t)
Drawdown_t = Equity_t / RunningPeak_t - 1
MaxDrawdown = min(Drawdown_t)
```

**实际含义**  
回撤 `-0.20` 表示当前净值比此前高点低 20%。最大回撤是整个区间最深的一次下跌。

**如何应用**

- 判断策略是否超出风险承受能力；
- 比较收益相近的策略；
- 计算卡玛比率；
- 设计减仓、止损和风险预算。

**图怎么看**  
0 表示净值正在创新高；负值越深表示离历史高点越远。还要看回撤持续时间：跌 20% 后一周修复，与三年未修复完全不同。

**和其他指标的关系**  
波动率描述日常抖动，最大回撤描述最坏路径，两者互补。卡玛比率用年化收益除以最大回撤。

**局限与误区**  
最大回撤只记录样本内最坏一次，不代表未来上限；它对起止区间和样本长度敏感。

In [ ]:
equity = 1 + df["cum_ret"]
df["drawdown"] = equity / equity.cummax() - 1
fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(df.index, df["drawdown"], 0, color="red", alpha=0.3, label="Drawdown")
ax.set_title("Drawdown")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()
df[["cum_ret", "drawdown"]].tail()

## 3. 趋势类指标：把噪声压平，看方向

每日价格充满噪声。趋势指标通过平滑或比较不同时间尺度，试图回答：

```text
价格总体向上、向下，还是没有明确方向？
```

趋势指标通常**滞后**。这是代价，不是 bug：只有看到一段数据后，才能确认趋势。它们更适合趋势市场，在横盘市场容易反复发出假信号。

### 3.1 SMA 简单移动平均线

**为什么需要它**  
原始价格每天跳动，很难看清中期方向。SMA 用最近 n 天平均价格过滤短期噪声。

**来源与计算**

```text
SMA_n(t) = mean(Close_{t-n+1} ... Close_t)
```

它给窗口内每一天相同权重。窗口向前移动，最老数据被移除，最新数据加入。

**实际含义**  
SMA20 是最近约一个交易月的平均持有成本代理；SMA60 约代表一季度。价格高于均线说明当前价格高于近期平均水平，不等同于“必涨”。

**如何应用**

- 均线方向向上：观察趋势；
- 价格上穿均线：可能发生状态变化；
- 短均线上穿长均线：常见趋势信号；
- 均线作为动态支撑/阻力的观察工具。

**图怎么看**  
短周期均线贴近价格、反应快；长周期均线更平滑、反应慢。交叉之后常常已经发生一段行情。

**关联与配合**  
BIAS 测量价格离 SMA 多远；布林带用 SMA 作中轨；BBI 是多条 SMA 的平均。可用 ADX 判断均线信号出现时是否真的存在趋势。

**局限与误区**  
SMA 是滞后指标，横盘期会频繁交叉。交叉本身不是盈利保证，必须加成本、止损和样本外验证。

In [ ]:
df["sma_20"] = df["Close"].rolling(20).mean()
df["sma_60"] = df["Close"].rolling(60).mean()
plot_lines(df, ["Close", "sma_20", "sma_60"], "SMA20 / SMA60")
df[["Close", "sma_20", "sma_60"]].tail()

### 3.2 EMA 指数移动平均线

**为什么需要它**  
SMA 对窗口内所有价格同权，而且旧数据移出窗口时可能产生跳变。EMA 让最新价格权重更高，反应更快。

**来源与计算**

```text
alpha = 2 / (n + 1)
EMA_t = alpha × Close_t + (1-alpha) × EMA_{t-1}
```

历史价格的权重按指数衰减，但不会在某一天突然归零。

**实际含义**  
EMA12 比 EMA26 更敏感。价格快速上涨时，短 EMA 通常先抬头。

**如何应用**

- 观察较灵敏的趋势；
- 比较快慢 EMA；
- 构造 MACD；
- 作为 Keltner Channel 中线。

**图怎么看**  
EMA 通常比同周期 SMA 更靠近价格。越灵敏也意味着越容易被噪声扰动。

**关联与配合**  
MACD 本质是快慢 EMA 的差；Keltner 使用 EMA + ATR。EMA 与 SMA 不应简单比较谁“更准”，而是响应速度和稳定性的权衡。

**局限与误区**  
EMA 仍然只使用历史价格，不能预知反转。周期越短，信号越快但假信号越多。

In [ ]:
df["ema_12"] = df["Close"].ewm(span=12, adjust=False).mean()
df["ema_26"] = df["Close"].ewm(span=26, adjust=False).mean()
plot_lines(df, ["Close", "ema_12", "ema_26"], "EMA12 / EMA26")
df[["Close", "ema_12", "ema_26"]].tail()

### 3.3 BIAS 乖离率 / 均线偏离

**为什么需要它**  
仅知道价格在均线上方还不够，还想知道“离得有多远”。

**来源与计算**

```text
BIAS_n = Close / SMA_n - 1
```

有些行情软件乘以 100，以百分数显示；本 Notebook 保留小数。

**实际含义**  
BIAS20 = 0.10 表示价格比 20 日均线高 10%；-0.08 表示低 8%。

**如何应用**

- 趋势策略：正 BIAS 表示价格位于均线上方；
- 均值回归：极端乖离可能意味着短期过热/过冷；
- 比较不同股票时，用比例比直接价格差更合理。

**图怎么看**  
围绕 0 波动。长期保持正值可能是强趋势，而不是立即卖出信号。

**关联与配合**  
BIAS 和布林 `%B` 都测量偏离；BIAS 不考虑当前波动水平，布林带用标准差做了波动调整。可配合 RSI 判断偏离是否伴随动量过热。

**局限与误区**  
没有通用的“BIAS 超过多少必反转”。阈值依赖股票、周期和波动环境。

In [ ]:
df["bias_20"] = df["Close"] / df["sma_20"] - 1
plot_price_with(df, ["bias_20"], "BIAS20 = Close / SMA20 - 1")
df[["Close", "sma_20", "bias_20"]].tail()

### 3.4 MACD 指数平滑异同移动平均线

**为什么需要它**  
两条均线交叉只能给出离散事件。MACD 把快慢趋势的距离连续化，既看方向，也看趋势动能是在增强还是减弱。

**来源**  
Gerald Appel 在 20 世纪 70 年代提出 MACD。核心是快 EMA 与慢 EMA 的差。

**计算**

```text
DIF = EMA12 - EMA26
DEA = EMA(DIF, 9)
Histogram = DIF - DEA
```

不同软件可能把柱体显示为 `2 × (DIF - DEA)`，使用时要确认口径。

**实际含义**

- DIF > 0：短期平均价格高于长期平均价格；
- DIF 上穿 DEA：趋势动能可能转强；
- 柱体扩大：DIF 与其平滑线距离增加；
- 价格创新高而 MACD 未创新高：称为背离，但不是确定反转。

**如何应用**  
用于趋势确认、动能变化、背离观察；常与均线、ADX、成交量配合。

**图怎么看**  
不要只看一次金叉。先看零轴位置，再看 DIF/DEA 方向，最后看柱体扩张或收缩。

**关联与配合**  
MACD 来自 EMA，因此也是滞后趋势指标。ADX 可补充趋势强度，成交量可验证突破参与度。

**局限与误区**  
震荡市场金叉死叉密集；背离可以持续很久。参数 12/26/9 是惯例，不是自然定律。

In [ ]:
df["macd_dif"] = df["ema_12"] - df["ema_26"]
df["macd_dea"] = df["macd_dif"].ewm(span=9, adjust=False).mean()
df["macd_hist"] = df["macd_dif"] - df["macd_dea"]

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(df.index, df["Close"], label="Close")
axes[0].grid(True, alpha=0.3); axes[0].legend(); axes[0].set_title("Price")
axes[1].plot(df.index, df["macd_dif"], label="DIF")
axes[1].plot(df.index, df["macd_dea"], label="DEA")
axes[1].bar(df.index, df["macd_hist"], alpha=0.35, label="Histogram")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].grid(True, alpha=0.3); axes[1].legend(); axes[1].set_title("MACD")
plt.tight_layout(); plt.show()
df[["macd_dif", "macd_dea", "macd_hist"]].tail()

### 3.5 BBI 多空指标

**为什么需要它**  
单条均线对周期很敏感。BBI 希望把短、中周期均线综合成一条更稳定的趋势线。

**计算**

```text
BBI = (MA3 + MA6 + MA12 + MA24) / 4
```

**实际含义**  
它可理解为多个时间尺度平均成本的再平均。价格在 BBI 上方通常表示综合趋势偏强。

**如何应用**  
观察价格与 BBI 的位置、BBI 斜率和穿越；适合做简单趋势过滤。

**图怎么看**  
BBI 比短均线平滑，比长均线灵敏。方向改变通常晚于价格。

**关联与配合**  
它属于均线家族，与 SMA/EMA 信息高度重叠，不应把它们同时当成多个独立确认信号。

**局限与误区**  
多条均线平均不会消除滞后，只是在灵敏和稳定之间折中。横盘期仍会反复穿越。

In [ ]:
df["bbi"] = (
    df["Close"].rolling(3).mean() +
    df["Close"].rolling(6).mean() +
    df["Close"].rolling(12).mean() +
    df["Close"].rolling(24).mean()
) / 4
plot_lines(df, ["Close", "bbi"], "BBI")
df[["Close", "bbi"]].tail()

## 4. 动量与震荡类指标：价格跑得多快、处在区间什么位置

趋势指标看方向，动量指标看速度，震荡指标看相对位置。它们常被用来寻找“过热/过冷”，但强趋势中可以长期超买或超卖，因此阈值不能机械解释为买卖指令。

### 4.1 ROC 变动率

**为什么需要它**  
趋势告诉我们方向，ROC 直接回答“过去 n 天涨了多少”。

**计算**

```text
ROC_n = Close_t / Close_{t-n} - 1
```

**实际含义**  
ROC20 = 0.15 表示过去 20 个交易日上涨 15%。它有方向，也有幅度。

**如何应用**

- 时间序列动量：ROC > 0 时认为趋势向上；
- 横截面动量：同一时点对股票排序；
- 观察动量加速或减速。

**图怎么看**  
0 是分界。值越大表示过去窗口涨幅越高，但不代表未来必涨。

**关联与配合**  
ROC 与收益率同源；MACD 也测量动量但先平滑；RSI 把上涨和下跌力度压缩到 0–100。

**局限与误区**  
窗口起点价格会突然移出，造成跳变。极高 ROC 可能是趋势，也可能是过热。

In [ ]:
df["roc_20"] = df["Close"].pct_change(20)
plot_price_with(df, ["roc_20"], "ROC20")
df[["Close", "roc_20"]].tail()

### 4.2 RSI 相对强弱指标

**为什么需要它**  
只看涨跌幅无法区分“连续小涨”和“一天暴涨”。RSI 比较一段时间平均上涨幅度与平均下跌幅度。

**来源**  
J. Welles Wilder 在 1978 年系统提出 RSI，标准周期常用 14。

**计算**

```text
RS = Wilder平均上涨幅度 / Wilder平均下跌幅度
RSI = 100 - 100 / (1 + RS)
```

本 Notebook 使用 `alpha=1/n` 的指数平滑，接近 Wilder 的 RMA。

**实际含义**  
RSI 范围 0–100。70/30 是常见参考，不是物理边界：70 以上表示近期上涨力度显著强于下跌力度。

**如何应用**

- 震荡市场寻找超买/超卖；
- 趋势过滤：强势股 RSI 常维持在 50 上方；
- 观察价格与 RSI 背离。

**图怎么看**  
除了 70/30，更应看 RSI 的中轴 50、持续区间和方向。

**关联与配合**  
MFI 可理解为加入成交量的 RSI；Stochastic RSI 是对 RSI 再做区间标准化；可用 ADX 区分趋势和震荡环境。

**局限与误区**  
RSI > 70 不等于立刻卖出。强趋势中“超买”往往代表强，而不是马上跌。

In [ ]:
def calc_rsi(close, n=14):
    diff = close.diff()
    gain = diff.clip(lower=0)
    loss = -diff.clip(upper=0)
    avg_gain = gain.ewm(alpha=1/n, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/n, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - 100 / (1 + rs)

df["rsi_14"] = calc_rsi(df["Close"], 14)
plot_price_with(df, ["rsi_14"], "RSI14")
plt.figure(figsize=(14, 4))
plt.plot(df.index, df["rsi_14"], label="RSI14")
plt.axhline(70, color="red", linestyle="--", label="70")
plt.axhline(30, color="green", linestyle="--", label="30")
plt.title("RSI14 with 70/30 levels")
plt.grid(True, alpha=0.3); plt.legend(); plt.show()
df[["rsi_14"]].tail()

### 4.3 KDJ 随机指标

**为什么需要它**  
我们不仅关心涨跌，还关心收盘价位于近期高低区间的什么位置。强势市场常收在区间上部，弱势市场常收在下部。

**来源与计算**  
KDJ 来自随机指标 Stochastic Oscillator，国内行情软件常加入 J 线：

```text
RSV = (Close - LowestLow_n) / (HighestHigh_n - LowestLow_n) × 100
K = 平滑(RSV)
D = 平滑(K)
J = 3K - 2D
```

**实际含义**  
K、D 通常在 0–100；J 放大 K 与 D 的差，可超出 0–100，因此更敏感也更嘈杂。

**如何应用**  
观察 K/D 交叉、20/80 区域和背离；更适合有边界的震荡市场。

**图怎么看**  
J 最快、K 次之、D 最慢。交叉频率很高，单独使用容易过度交易。

**关联与配合**  
Williams %R 与 RSV 使用同一高低区间信息，只是尺度和方向不同；RSI 使用涨跌幅信息，因此两者有相关但不相同。

**局限与误区**  
趋势行情中会长期钝化在高位或低位。J 线极端值不是必然反转信号。

In [ ]:
low_n = df["Low"].rolling(9).min()
high_n = df["High"].rolling(9).max()
df["kdj_rsv"] = (df["Close"] - low_n) / (high_n - low_n) * 100
df["kdj_k"] = df["kdj_rsv"].ewm(alpha=1/3, adjust=False).mean()
df["kdj_d"] = df["kdj_k"].ewm(alpha=1/3, adjust=False).mean()
df["kdj_j"] = 3 * df["kdj_k"] - 2 * df["kdj_d"]
plot_price_with(df, ["kdj_k", "kdj_d", "kdj_j"], "KDJ")
df[["kdj_k", "kdj_d", "kdj_j"]].tail()

### 4.4 CCI 顺势指标

**为什么需要它**  
仅看价格离均线多少，没有考虑该资产平时的偏离尺度。CCI 用平均绝对偏差对价格偏离做标准化。

**来源**  
Donald Lambert 在 1980 年提出，最初用于商品周期分析，后来广泛用于股票。

**计算**

```text
TP = (High + Low + Close) / 3
MA = mean(TP, n)
MeanDeviation = mean(abs(TP_i - MA), n)
CCI = (TP - MA) / (0.015 × MeanDeviation)
```

常数 0.015 使大部分值落在约 -100 到 +100 之间。本 Notebook 已按每个窗口内同一均值正确计算 Mean Deviation。

**实际含义**  
CCI > 100 表示典型价格明显高于近期正常水平；CCI < -100 表示明显偏低。

**如何应用**  
既可用于趋势突破（越过 +100），也可用于均值回归（极端后回归），两种逻辑相反，必须先明确策略假设。

**图怎么看**  
看 ±100 区域、持续时间和与价格的背离。

**关联与配合**  
CCI、BIAS、布林带都测偏离：BIAS 用比例，布林带用标准差，CCI 用平均绝对偏差和典型价格。

**局限与误区**  
不同资产的阈值表现不同；极端值可在强趋势中继续扩大。

In [ ]:
tp = (df["High"] + df["Low"] + df["Close"]) / 3
tp_ma = tp.rolling(20).mean()
# 每个窗口内，所有 TP 都相对于该窗口的同一个均值计算平均绝对偏差。
mad = tp.rolling(20).apply(lambda x: np.mean(np.abs(x - x.mean())), raw=True)
df["cci_20"] = (tp - tp_ma) / (0.015 * mad)

last = tp.dropna().iloc[-20:].to_numpy()
assert np.isclose(mad.dropna().iloc[-1], np.mean(np.abs(last - last.mean())))

plot_price_with(df, ["cci_20"], "CCI20")
plt.figure(figsize=(14, 4))
plt.plot(df.index, df["cci_20"], label="CCI20")
plt.axhline(100, color="red", linestyle="--")
plt.axhline(-100, color="green", linestyle="--")
plt.title("CCI20 with +/-100 levels")
plt.grid(True, alpha=0.3); plt.legend(); plt.show()
df[["cci_20"]].tail()

### 4.5 Williams %R 威廉指标

**为什么需要它**  
它快速判断当前收盘价离近期最高价还有多远。

**来源与计算**  
由 Larry Williams 推广：

```text
%R = (HighestHigh_n - Close) / (HighestHigh_n - LowestLow_n) × -100
```

**实际含义**  
通常位于 -100 到 0。接近 0 表示收盘靠近近期高位，接近 -100 表示靠近近期低位。常见参考为 -20 与 -80。

**如何应用**  
震荡行情观察极端位置，或观察指标离开极端区域的时点。

**图怎么看**  
注意坐标方向：-10 比 -90 更“高”。

**关联与配合**  
它与 KDJ 的 RSV 几乎是同一信息的反向尺度；同时使用二者不会带来两个独立证据。

**局限与误区**  
趋势市场会长期停留在极端区。进入 -20 不代表上涨结束，进入 -80 也不代表下跌结束。

In [ ]:
hh = df["High"].rolling(14).max()
ll = df["Low"].rolling(14).min()
df["willr_14"] = (hh - df["Close"]) / (hh - ll) * -100
plot_price_with(df, ["willr_14"], "Williams %R 14")
df[["willr_14"]].tail()

## 5. 波动与通道类指标：价格会动多远，是否突破正常范围

方向和风险是两件事：股票可以缓慢上涨，也可以剧烈震荡后原地踏步。波动指标测量变化幅度；通道指标把“正常范围”画到价格图上。

### 5.1 历史波动率 Historical Volatility

**为什么需要它**  
收益相同的两只股票，路径可能完全不同。波动率衡量日收益的不稳定程度。

**计算**

```text
DailyVol_n = std(Return, n)
AnnualVol_n = DailyVol_n × sqrt(252)
```

平方根年化假设日收益相对独立且方差可加。

**实际含义**  
年化波动率 30% 不是“预计一年涨跌 30%”，而是日收益标准差换算到年度尺度。

**如何应用**

- 比较资产风险；
- 波动率过滤和目标波动仓位；
- 计算夏普；
- 识别市场风险状态切换。

**图怎么看**  
波动率常有聚集性：高波动之后仍可能高波动。方向未知，涨停和跌停都会提高波动率。

**关联与配合**  
布林带宽度来自价格标准差；ATR 用价格范围而非收益标准差；NATR 可跨价格水平比较。

**局限与误区**  
历史波动率向后看且对极端值敏感；它把上涨和下跌同等视为风险，也不能描述尾部形状。

In [ ]:
df["vol_20"] = df["ret_1d"].rolling(20).std() * np.sqrt(252)
plot_price_with(df, ["vol_20"], "Annualized Historical Volatility 20")
df[["vol_20"]].tail()

### 5.2 Bollinger Bands 布林带

**为什么需要它**  
固定百分比通道无法适应市场波动变化。布林带让通道在高波动时变宽、低波动时收窄。

**来源与计算**  
John Bollinger 在 1980 年代发展此指标：

```text
Middle = SMA20
Upper = Middle + 2 × STD20
Lower = Middle - 2 × STD20
%B = (Close - Lower) / (Upper - Lower)
BandWidth = (Upper - Lower) / Middle
```

本 Notebook 使用总体标准差 `ddof=0`；软件之间可能有口径差异。

**实际含义**  
价格触上轨表示相对近期均值处于高位，不是自动卖点。带宽反映近期波动收缩或扩张。

**如何应用**

- 均值回归：极端偏离后回归中轨；
- 突破：低带宽挤压后放量突破；
- `%B` 标准化价格在带内的位置。

**图怎么看**  
先看带宽状态，再看价格是否沿上轨/下轨运行。强趋势中价格可以“贴着上轨走”。

**关联与配合**  
BIAS 不调整波动，布林带会；Keltner 用 ATR 替代标准差。布林挤压常配合成交量和 ADX 确认。

**局限与误区**  
“两倍标准差”不保证 95% 价格都在带内，因为金融收益并非稳定正态分布。

In [ ]:
mid = df["Close"].rolling(20).mean()
std = df["Close"].rolling(20).std(ddof=0)
df["bb_mid"] = mid
df["bb_upper"] = mid + 2 * std
df["bb_lower"] = mid - 2 * std
df["bb_width"] = (df["bb_upper"] - df["bb_lower"]) / df["bb_mid"]
df["bb_percent_b"] = (df["Close"] - df["bb_lower"]) / (df["bb_upper"] - df["bb_lower"])

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df["Close"], label="Close")
ax.plot(df.index, df["bb_mid"], label="Middle")
ax.plot(df.index, df["bb_upper"], label="Upper")
ax.plot(df.index, df["bb_lower"], label="Lower")
ax.fill_between(df.index, df["bb_lower"], df["bb_upper"], alpha=0.12)
ax.set_title("Bollinger Bands")
ax.grid(True, alpha=0.3); ax.legend(); plt.show()
df[["bb_mid", "bb_upper", "bb_lower", "bb_width", "bb_percent_b"]].tail()

### 5.3 ATR 平均真实波幅

**为什么需要它**  
只看 `High-Low` 会漏掉隔夜跳空。ATR 把前收盘价纳入，测量真实价格波动范围。

**来源**  
Welles Wilder 在 1978 年提出 True Range 与 ATR。

**计算**

```text
TR = max(
    High - Low,
    abs(High - PreviousClose),
    abs(Low - PreviousClose)
)
ATR_n = Wilder平滑(TR, n)
NATR_n = ATR_n / Close
```

**实际含义**  
ATR14 = 0.50 表示最近真实日波幅约 0.50 元。NATR 除以价格后可跨股票比较。

**如何应用**

- 设置基于波动的止损，例如 2×ATR；
- 按波动调整仓位；
- 构造 Keltner Channel；
- 判断行情是否开始扩张。

**图怎么看**  
ATR 上升说明波动扩大，不表示上涨；价格高的股票 ATR 天然更大，所以跨标的看 NATR。

**关联与配合**  
ATR、历史波动率都测波动，但 ATR 基于价格区间并包含跳空，历史波动率基于收益标准差。

**局限与误区**  
ATR 没有方向，不能单独给出买卖信号。参数和平滑口径在不同软件中可能不同。

In [ ]:
prev_close = df["Close"].shift(1)
tr = pd.concat([
    df["High"] - df["Low"],
    (df["High"] - prev_close).abs(),
    (df["Low"] - prev_close).abs(),
], axis=1).max(axis=1)
df["tr"] = tr
df["atr_14"] = tr.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
df["natr_14"] = df["atr_14"] / df["Close"]
plot_price_with(df, ["atr_14", "natr_14"], "ATR14 and NATR14")
df[["tr", "atr_14", "natr_14"]].tail()

### 5.4 Donchian Channel 唐奇安通道

**为什么需要它**  
趋势交易者想知道价格是否突破了过去一段时间的边界。

**来源与计算**  
Richard Donchian 推广的经典趋势跟踪通道：

```text
Upper_n = rolling_max(High, n)
Lower_n = rolling_min(Low, n)
Middle_n = (Upper_n + Lower_n) / 2
```

**实际含义**  
上轨是过去 n 天最高价，下轨是最低价。新高突破说明价格离开原有区间。

**如何应用**  
突破上轨入场、跌破下轨退出；海龟交易思想与此类规则关系密切。

**图怎么看**  
通道会呈阶梯状。注意实盘信号应使用前一日通道，否则当天最高价包含当前信息，容易产生同柱自引用。

**关联与配合**  
可用 ADX 过滤无趋势突破，用成交量确认参与度，用 ATR 设置止损。

**局限与误区**  
横盘期假突破多；窗口越短越敏感，越长越迟钝。直接用当天 rolling max 做当天成交信号需防未来函数。

In [ ]:
df["donchian_upper_20"] = df["High"].rolling(20).max()
df["donchian_lower_20"] = df["Low"].rolling(20).min()
df["donchian_mid_20"] = (df["donchian_upper_20"] + df["donchian_lower_20"]) / 2
plot_lines(df, ["Close", "donchian_upper_20", "donchian_mid_20", "donchian_lower_20"], "Donchian Channel 20")
df[["donchian_upper_20", "donchian_mid_20", "donchian_lower_20"]].tail()

### 5.5 Keltner Channel 肯特纳通道

**为什么需要它**  
希望同时考虑趋势中心和真实波动，并获得比布林带更平滑的通道。

**来源与计算**  
现代常见版本使用 EMA 与 ATR：

```text
Middle = EMA20
Upper = EMA20 + multiplier × ATR20
Lower = EMA20 - multiplier × ATR20
```

原始 Keltner 公式与现代版本不同，使用时应说明口径。

**实际含义**  
中线代表趋势，通道宽度代表真实波幅。价格越过上轨表示相对当前趋势和波动处于强势位置。

**如何应用**  
趋势突破、回踩中线、动态止损；也常与布林带比较构造 squeeze。

**图怎么看**  
看价格是否沿通道运行，以及通道方向和宽度是否同步变化。

**关联与配合**  
布林带 = SMA + 标准差；Keltner = EMA + ATR。两者差异来自中心和平滑尺度。

**局限与误区**  
倍数 2 只是常见设置；不同资产和周期需要验证。通道突破也可能是假信号。

In [ ]:
ema20 = df["Close"].ewm(span=20, adjust=False).mean()
atr20 = df["tr"].ewm(alpha=1/20, adjust=False, min_periods=20).mean()
df["kc_mid"] = ema20
df["kc_upper"] = ema20 + 2 * atr20
df["kc_lower"] = ema20 - 2 * atr20
plot_lines(df, ["Close", "kc_upper", "kc_mid", "kc_lower"], "Keltner Channel")
df[["kc_upper", "kc_mid", "kc_lower"]].tail()

## 6. 趋势强度与止损指标：不仅问方向，还问趋势是否值得跟

均线上扬不代表趋势足够强。DMI/ADX 把方向和强度拆开；Parabolic SAR 把趋势跟踪转成随行情移动的止损参考。

### 6.1 DMI / ADX

**为什么需要它**  
均线能看方向，但很难区分“强趋势”和“轻微漂移”。DMI 测量上涨/下跌方向移动，ADX 再测量两者分离程度。

**来源**  
Welles Wilder 在 1978 年提出，与 RSI、ATR 同属一套指标体系。

**计算直觉**

```text
+DM：今天最高价向上扩张多少
-DM：今天最低价向下扩张多少
+DI = Wilder平滑(+DM) / Wilder平滑(TR) × 100
-DI = Wilder平滑(-DM) / Wilder平滑(TR) × 100
DX = abs(+DI - -DI) / (+DI + -DI) × 100
ADX = Wilder平滑(DX)
```

**实际含义**

- +DI > -DI：上涨方向力量更强；
- -DI > +DI：下跌方向力量更强；
- ADX 上升：趋势强度增加，不代表方向向上；
- ADX 低：更可能处于震荡。

**如何应用**  
用 DI 判断方向、ADX 过滤趋势策略。常见 ADX 20/25 只是经验线。

**图怎么看**  
必须把 ADX 和 ±DI 一起看。ADX 高但 -DI 占优，可能是强下跌趋势。

**关联与配合**  
ATR 是 DMI 的分母；ADX 可过滤均线、MACD、Donchian 信号。

**局限与误区**  
ADX 本身不区分涨跌；平滑使其滞后。不同库的 Wilder 平滑和初始化方式可能略有不同。

In [ ]:
up_move = df["High"].diff()
down_move = -df["Low"].diff()
plus_dm = pd.Series(np.where((up_move > down_move) & (up_move > 0), up_move, 0.0), index=df.index)
minus_dm = pd.Series(np.where((down_move > up_move) & (down_move > 0), down_move, 0.0), index=df.index)

# Wilder RMA：alpha=1/n。先分别平滑方向移动与真实波幅，再求 DI。
smoothed_tr = df["tr"].ewm(alpha=1/14, adjust=False, min_periods=14).mean()
smoothed_plus_dm = plus_dm.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
smoothed_minus_dm = minus_dm.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
df["plus_di_14"] = 100 * smoothed_plus_dm / smoothed_tr
df["minus_di_14"] = 100 * smoothed_minus_dm / smoothed_tr
dx = (df["plus_di_14"] - df["minus_di_14"]).abs() / (df["plus_di_14"] + df["minus_di_14"]).replace(0, np.nan) * 100
df["adx_14"] = dx.ewm(alpha=1/14, adjust=False, min_periods=14).mean()
plot_price_with(df, ["plus_di_14", "minus_di_14", "adx_14"], "DMI / ADX 14")
df[["plus_di_14", "minus_di_14", "adx_14"]].tail()

### 6.2 Parabolic SAR 抛物线转向

**为什么需要它**  
趋势持仓最难的是“何时退出”。SAR 设计成会逐渐靠近价格的移动止损点，让盈利趋势有空间，同时逐步收紧退出线。

**来源与计算**  
同样由 Wilder 提出：

```text
SAR_next = SAR_current + AF × (EP - SAR_current)
```

`EP` 是当前趋势的极值，`AF` 是加速因子。每出现新极值，AF 增加，SAR 更快靠近价格；反转后重新开始。

**实际含义**  
点在价格下方时通常表示多头状态，在价格上方时表示空头状态。价格穿越 SAR 时发生状态翻转。

**如何应用**  
趋势跟踪退出、移动止损、反转提示。更适合方向明显的行情。

**图怎么看**  
观察点位从价格下方翻到上方，或从上方翻到下方；连续趋势中点位逐渐靠近价格。

**关联与配合**  
SAR 提供退出位置，ADX 判断是否有趋势，均线/MACD 判断方向。

**局限与误区**  
震荡市场会频繁翻转。不同实现的初始化和极值处理可能不同；本 Notebook 是教学实现，生产使用应与目标交易软件对齐。

In [ ]:
def parabolic_sar(high, low, step=0.02, max_step=0.2):
    index = getattr(high, "index", pd.RangeIndex(len(high)))
    high = np.asarray(high)
    low = np.asarray(low)
    n = len(high)
    sar = np.full(n, np.nan)
    bull = True
    af = step
    ep = high[0]
    sar[0] = low[0]
    for i in range(1, n):
        prev_sar = sar[i-1]
        sar[i] = prev_sar + af * (ep - prev_sar)
        if bull:
            sar[i] = min(sar[i], low[i-1], low[i-2] if i > 1 else low[i-1])
            if low[i] < sar[i]:
                bull = False
                sar[i] = ep
                ep = low[i]
                af = step
            elif high[i] > ep:
                ep = high[i]
                af = min(af + step, max_step)
        else:
            sar[i] = max(sar[i], high[i-1], high[i-2] if i > 1 else high[i-1])
            if high[i] > sar[i]:
                bull = True
                sar[i] = ep
                ep = high[i]
                af = step
            elif low[i] < ep:
                ep = low[i]
                af = min(af + step, max_step)
    return pd.Series(sar, index=index)

df["sar"] = parabolic_sar(df["High"], df["Low"])
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(df.index, df["Close"], label="Close")
ax.scatter(df.index, df["sar"], s=8, label="SAR", color="red", alpha=0.7)
ax.set_title("Parabolic SAR")
ax.grid(True, alpha=0.3); ax.legend(); plt.show()
df[["Close", "sar"]].tail()

## 7. 成交量、资金流与流动性指标：价格变化背后有多少参与

价格上升可以发生在巨量成交中，也可以发生在几乎无人交易时。量价指标尝试回答：

```text
这次涨跌有没有成交量支持？资金更像流入还是流出？大资金是否真的能成交？
```

注意：成交量不等于主动买入资金。每一笔成交同时有买方和卖方，“资金流入/流出”只是依据成交位置构造的代理指标。

### 7.1 成交量均线与成交量放大倍数

**为什么需要它**  
单看今天成交量没有参照。需要和该股票自己的近期正常水平比较。

**计算**

```text
VolumeMA20 = mean(Volume, 20)
VolumeRatio20 = Volume / VolumeMA20
```

**实际含义**  
量比 1.5 表示今天成交量是近 20 日平均的 1.5 倍；小于 1 表示缩量。

**如何应用**  
确认突破、观察恐慌放量、过滤流动性不足的信号。

**图怎么看**  
成交量柱与均量线一起看；孤立巨量和持续放量意义不同。

**关联与配合**  
与 Donchian/布林突破、MACD 趋势信号配合；Amount 和 Turnover 可补充价格与流通盘差异。

**局限与误区**  
放量只表示分歧和交易活跃，不表示必涨。不同市场/接口的 Volume 单位可能是股或手，使用前要确认。

In [ ]:
df["volume_ma_20"] = df["Volume"].rolling(20).mean()
df["volume_ratio_20"] = df["Volume"] / df["volume_ma_20"]
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].bar(df.index, df["Volume"], alpha=0.35, label="Volume")
axes[0].plot(df.index, df["volume_ma_20"], color="red", label="Volume MA20")
axes[0].grid(True, alpha=0.3); axes[0].legend(); axes[0].set_title("Volume and MA20")
axes[1].plot(df.index, df["volume_ratio_20"], label="Volume Ratio 20")
axes[1].axhline(1, color="black", linewidth=1)
axes[1].grid(True, alpha=0.3); axes[1].legend(); axes[1].set_title("Volume Ratio")
plt.tight_layout(); plt.show()
df[["Volume", "volume_ma_20", "volume_ratio_20"]].tail()

### 7.2 OBV 能量潮

**为什么需要它**  
希望把成交量按价格方向累计，观察量能是否先于价格发生变化。

**来源**  
Joseph Granville 在 1960 年代推广 OBV。

**计算**

```text
上涨日：OBV_t = OBV_{t-1} + Volume_t
下跌日：OBV_t = OBV_{t-1} - Volume_t
平盘日：OBV_t = OBV_{t-1}
```

**实际含义**  
OBV 的绝对值没有直接意义，主要看趋势和与价格是否同步。

**如何应用**  
价格上涨且 OBV 同步创新高，量价较一致；价格创新高而 OBV 未创新高，可视为背离线索。

**图怎么看**  
比较价格和 OBV 的方向、峰谷，不比较数值单位。

**关联与配合**  
PVT 进一步考虑涨跌幅大小；ADL 根据收盘在日内区间的位置分配成交量。

**局限与误区**  
一天只根据收盘涨跌给全部成交量同一符号，信息粗糙；单日巨量会永久改变累计曲线。

In [ ]:
price_direction = np.sign(df["Close"].diff()).fillna(0)
df["obv"] = (price_direction * df["Volume"]).cumsum()
plot_price_with(df, ["obv"], "OBV")
df[["Close", "Volume", "obv"]].tail()

### 7.3 PVT 价量趋势指标

**为什么需要它**  
OBV 认为上涨 0.1% 和上涨 10% 都应该加同样成交量。PVT 用收益率给成交量加权。

**计算**

```text
PVT_t = PVT_{t-1} + Volume_t × Return_t
```

**实际含义**  
大涨且放量会显著推高 PVT，大跌且放量会显著压低 PVT。

**如何应用**  
观察价量趋势同步和背离；不常直接使用固定阈值。

**图怎么看**  
重点看方向、突破和峰谷是否确认价格走势。

**关联与配合**  
OBV 只看方向，PVT 同时看方向与涨跌幅；两者仍然都是累计指标。

**局限与误区**  
对异常收益和异常成交量敏感；绝对数值依赖成交量单位，不能跨股票直接比较。

In [ ]:
df["pvt"] = (df["Volume"] * df["ret_1d"].fillna(0)).cumsum()
plot_price_with(df, ["pvt"], "PVT")
df[["ret_1d", "Volume", "pvt"]].tail()

### 7.4 ADL 累积/派发线

**为什么需要它**  
仅根据“今天涨没涨”分配成交量太粗。ADL 看收盘价更靠近当天最高还是最低，以估计买卖压力。

**计算**

```text
MFM = ((Close-Low) - (High-Close)) / (High-Low)
MFV = MFM × Volume
ADL = cumulative_sum(MFV)
```

MFM 接近 +1 表示收盘接近最高，接近 -1 表示收盘接近最低。

**实际含义**  
ADL 上升表示成交量更多发生在偏强收盘日；下降表示偏弱。

**如何应用**  
观察趋势确认和量价背离。

**图怎么看**  
对比价格与 ADL 的新高、新低和方向。

**关联与配合**  
CMF 是 ADL 资金流思想的滚动窗口版本；OBV 用日涨跌方向，ADL 用日内收盘位置。

**局限与误区**  
只用日线 OHLC 近似日内资金流，无法知道真实主动买卖；跳空行情可能产生不直观结果。

In [ ]:
mfm = ((df["Close"] - df["Low"]) - (df["High"] - df["Close"])) / (df["High"] - df["Low"]).replace(0, np.nan)
df["mfv"] = mfm * df["Volume"]
df["adl"] = df["mfv"].fillna(0).cumsum()
plot_price_with(df, ["adl"], "Accumulation/Distribution Line")
df[["mfv", "adl"]].tail()

### 7.5 CMF Chaikin Money Flow

**为什么需要它**  
ADL 是从起点累计，早期数据会长期影响结果。CMF 只看最近窗口，更容易判断当前资金流状态。

**来源与计算**  
由 Marc Chaikin 发展：

```text
CMF_n = sum(MFV, n) / sum(Volume, n)
```

其中 MFV 来自 ADL 的 Money Flow Volume。

**实际含义**  
通常位于 -1 到 +1。正值表示窗口内偏向“收集”，负值偏向“派发”。

**如何应用**  
趋势确认、突破确认、观察 0 轴切换。

**图怎么看**  
围绕 0 波动。持续正值比一次短暂穿越更有意义。

**关联与配合**  
CMF 与 ADL 同源；MFI 使用典型价格涨跌区分正负资金流，并压缩到 0–100。

**局限与误区**  
它仍是 OHLCV 的代理，不是真实订单流。窗口和阈值需要针对市场验证。

In [ ]:
df["cmf_20"] = df["mfv"].rolling(20).sum() / df["Volume"].rolling(20).sum()
plot_price_with(df, ["cmf_20"], "CMF20")
df[["cmf_20"]].tail()

### 7.6 MFI 资金流量指标

**为什么需要它**  
RSI 只看价格涨跌力度，MFI 希望把成交量也纳入，回答“价格强弱是否伴随资金参与”。

**计算**

```text
TypicalPrice = (High + Low + Close) / 3
RawMoneyFlow = TypicalPrice × Volume
MoneyRatio = PositiveMoneyFlow / NegativeMoneyFlow
MFI = 100 - 100 / (1 + MoneyRatio)
```

按典型价格相对前一日的涨跌，把资金流分成正负。

**实际含义**  
范围 0–100，常见参考 80/20。高值表示近期正资金流占优。

**如何应用**  
超买超卖、背离、价格信号的成交量确认。

**图怎么看**  
和 RSI 类似，但变化还受成交量影响。

**关联与配合**  
MFI 常被称为“带成交量的 RSI”；CMF 则基于收盘在日内区间的位置。

**局限与误区**  
成交量大并不等于资金净流入，MFI 仍然只是代理。强趋势中也会长期停留极端区。

In [ ]:
typical_price = (df["High"] + df["Low"] + df["Close"]) / 3
money_flow = typical_price * df["Volume"]
pos_flow = money_flow.where(typical_price > typical_price.shift(1), 0)
neg_flow = money_flow.where(typical_price < typical_price.shift(1), 0)
mfr = pos_flow.rolling(14).sum() / neg_flow.rolling(14).sum()
df["mfi_14"] = 100 - 100 / (1 + mfr)
plot_price_with(df, ["mfi_14"], "MFI14")
df[["mfi_14"]].tail()

### 7.7 VWAP 成交量加权平均价格

**为什么需要它**  
普通均线让每个时间点同权，但市场实际成交并不均匀。VWAP 让成交量大的价格权重更高。

**标准概念**

```text
VWAP = sum(Price_i × Volume_i) / sum(Volume_i)
```

机构通常使用**日内逐笔或分钟数据**计算从开盘开始累计的 Session VWAP。

**本 Notebook 的口径**  
这里只用日线数据，因此计算的是最近 20 日典型价格的**滚动成交量加权均价代理**：

```text
RollingVWAP20 = sum(TypicalPrice × Volume, 20) / sum(Volume, 20)
```

它不是严格的日内 VWAP。

**实际含义与应用**  
价格高于滚动 VWAP 表示高于近期成交量加权成本。真实日内 VWAP 常用于衡量执行价格好坏。

**图怎么看**  
比较 Close、SMA20 和 Rolling VWAP20；量大的交易日会更明显拉动 VWAP。

**关联与配合**  
SMA 按时间等权，VWAP 按成交量加权。若要做日内执行研究，应改用分钟/逐笔数据。

**局限与误区**  
不要把日线滚动 VWAP 当作交易软件里的当日 VWAP；两者数据粒度和重置方式不同。

In [ ]:
df["vwap_20"] = (typical_price * df["Volume"]).rolling(20).sum() / df["Volume"].rolling(20).sum()
plot_lines(df, ["Close", "vwap_20", "sma_20"], "VWAP20 vs SMA20")
df[["Close", "vwap_20", "sma_20"]].tail()

### 7.8 VR 成交量比率

**为什么需要它**  
想直接比较一段时间上涨日成交量与下跌日成交量谁更大。

**简化计算**

```text
VR_n = sum(UpVolume, n) / sum(DownVolume, n) × 100
```

部分软件会把平盘成交量的一半分别加入分子和分母，本 Notebook 使用不含平盘量的简化口径。

**实际含义**  
VR = 150 表示窗口内上涨日成交量总和约为下跌日的 1.5 倍。

**如何应用**  
观察市场热度、量能失衡和极端状态。

**图怎么看**  
看长期区间和异常峰值，不要把某个固定阈值当成必然买卖点。

**关联与配合**  
OBV 把量累计，VR 做窗口比值；MFI 还考虑典型价格。

**局限与误区**  
当下跌日成交量接近 0 时 VR 会爆大；不同软件公式口径可能不同。

In [ ]:
up_vol = df["Volume"].where(df["Close"] > df["Close"].shift(1), 0)
down_vol = df["Volume"].where(df["Close"] < df["Close"].shift(1), 0)
df["vr_26"] = up_vol.rolling(26).sum() / down_vol.rolling(26).sum().replace(0, np.nan) * 100
plot_price_with(df, ["vr_26"], "VR26")
df[["vr_26"]].tail()

### 7.9 Amihud 非流动性指标

**为什么需要它**  
成交量大不一定代表容易交易；更关键的是“一定成交额会推动价格变化多少”。

**来源**  
Yakov Amihud 在 2002 年提出广泛使用的非流动性代理：

```text
ILLIQ_t = abs(Return_t) / Amount_t
ILLIQ_period = mean(ILLIQ_t)
```

**实际含义**  
值越大，单位成交额对应的价格变化越大，流动性越差。其绝对值很小且有单位，实践中常乘 `10^6` 或 `10^8` 方便展示。

**如何应用**  
股票池流动性过滤、容量分析、小盘股研究、解释交易成本差异。

**图怎么看**  
观察滚动均值的抬升；流动性恶化时指标上升。

**关联与配合**  
与 Amount、Turnover、Bid-Ask Spread 共同描述流动性。回测时还应加入滑点和成交量约束。

**局限与误区**  
日线代理无法替代盘口价差和市场冲击模型；不同货币与成交额单位下不能直接比较原始数值。

In [ ]:
df["amihud"] = df["ret_1d"].abs() / df["Amount"].replace(0, np.nan)
df["amihud_scaled"] = df["amihud"] * 1e8
df["amihud_scaled_ma_20"] = df["amihud_scaled"].rolling(20).mean()
plot_price_with(df, ["amihud_scaled_ma_20"], "Amihud Illiquidity MA20 (×1e8)")
df[["Amount", "amihud", "amihud_scaled", "amihud_scaled_ma_20"]].tail()

## 8. 风险与绩效指标：不是用来择时，而是评价“赚得值不值”

技术指标通常从行情生成信号，风险绩效指标通常评价一段资产或策略收益。它们回答：赚了多少、过程多抖、最坏多惨、收益是否值得承担这些风险。

### 8.1 年化收益、年化波动、夏普、最大回撤与卡玛

#### 为什么需要这些指标

只看总收益无法公平比较：策略 A 两年赚 30%，策略 B 十年赚 30%，显然不同；同样年化收益下，路径更稳定的策略通常更容易持有。

#### 年化收益

```text
AnnualReturn = (1 + TotalReturn)^(252/N) - 1
```

表示按当前复利速度换算为一年大约增长多少。它不是每年实际都能获得的固定收益。

#### 年化波动

```text
AnnualVol = std(DailyReturn) × sqrt(252)
```

表示日收益波动换算到年度尺度。它对上涨和下跌一视同仁。

#### 夏普比率 Sharpe Ratio

William Sharpe 最初称其为 reward-to-variability ratio。它要回答：

```text
每承担一单位总波动，获得了多少超额收益？
```

标准日频样本估计：

```text
DailyExcess = DailyReturn - DailyRiskFreeRate
Sharpe = mean(DailyExcess) / std(DailyExcess) × sqrt(252)
```

直觉示例：两个策略都年化 10%，一个年化波动 8%，另一个 30%，前者夏普通常更高。

粗略经验（不能跨市场机械套用）：

| 夏普 | 常见直觉 |
|---:|---|
| < 0 | 连无风险收益都没补偿 |
| 0–1 | 收益风险比一般 |
| 1–2 | 较好，但要检查样本和成本 |
| > 2 | 很好，也要警惕过拟合或估值平滑 |

#### 最大回撤与卡玛比率

```text
MaxDrawdown = min(Equity / RunningPeak - 1)
Calmar = AnnualReturn / abs(MaxDrawdown)
```

卡玛更关注下行路径，而夏普关注全部波动。

#### 如何应用

- 同一市场、同一频率、相近区间内比较策略；
- 夏普配合最大回撤和卡玛；
- 同时检查收益是否集中在少数几天；
- 检查费用、滑点和样本外表现后再评价。

#### 关联与局限

夏普依赖收益分布近似稳定，无法充分描述偏度、肥尾和流动性风险。卖期权类策略可能长期小赚、偶尔巨亏，却在崩盘前显示较高夏普。高夏普不等于低最大回撤，更不等于未来稳定。

In [ ]:
def perf_metrics(returns, risk_free_annual=0.02):
    r = returns.dropna()
    n = len(r)
    total = (1 + r).prod() - 1
    ann_ret = (1 + total) ** (252 / n) - 1
    ann_vol = r.std(ddof=1) * np.sqrt(252)

    risk_free_daily = (1 + risk_free_annual) ** (1 / 252) - 1
    excess = r - risk_free_daily
    sharpe = excess.mean() / excess.std(ddof=1) * np.sqrt(252)

    downside = excess.clip(upper=0)
    downside_dev = np.sqrt((downside ** 2).mean()) * np.sqrt(252)
    sortino = excess.mean() * 252 / downside_dev if downside_dev > 0 else np.nan

    equity = (1 + r).cumprod()
    drawdown = equity / equity.cummax() - 1
    max_drawdown = drawdown.min()
    calmar = ann_ret / abs(max_drawdown) if max_drawdown < 0 else np.nan

    return pd.Series({
        "total_return": total,
        "annual_return": ann_ret,
        "annual_volatility": ann_vol,
        "risk_free_annual": risk_free_annual,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_drawdown,
        "calmar": calmar,
    })

perf = perf_metrics(df["ret_1d"])
assert np.isfinite(perf[["annual_return", "annual_volatility", "sharpe", "max_drawdown"]]).all()
perf

### 8.2 滚动夏普 Rolling Sharpe

**为什么需要它**  
全样本夏普把多年压成一个数字，会掩盖“前几年有效、后来失效”。滚动夏普观察收益风险比随时间变化。

**计算**

```text
RollingSharpe_n = mean(DailyExcessReturn, n)
                  / std(DailyExcessReturn, n)
                  × sqrt(252)
```

本例窗口为 60 个交易日，约三个月。

**实际含义**  
正值表示窗口内平均超额收益为正；负值表示表现弱于无风险基准。极端值可能来自波动率很小，必须同时看收益和波动。

**如何应用**

- 判断策略稳定性和可能的失效期；
- 比较不同市场状态；
- 辅助风险预算，而非每天根据其买卖。

**图怎么看**  
看是否长期为正、是否频繁跨零、是否只在某一小段极高。

**关联与配合**  
与滚动收益、滚动波动、回撤一起看，才能知道夏普变化来自收益还是风险。

**局限与误区**  
窗口重叠导致相邻值高度相关；60 日样本很少，估计噪声大。窗口越短越灵敏，也越不稳定。

In [ ]:
risk_free_daily = (1 + 0.02) ** (1 / 252) - 1
rolling_excess = df["ret_1d"] - risk_free_daily
df["rolling_sharpe_60"] = (
    rolling_excess.rolling(60).mean()
    / rolling_excess.rolling(60).std(ddof=1)
    * np.sqrt(252)
)
plot_price_with(df, ["rolling_sharpe_60"], "Rolling Sharpe 60")
df[["rolling_sharpe_60"]].tail()

### 8.3 VaR 与 CVaR：看收益分布的左尾

**为什么需要它**  
波动率描述整体分散程度，但投资者更关心极端亏损。VaR/CVaR 专注收益分布左尾。

**历史模拟法计算**

```text
VaR_95 = 日收益的 5% 分位数
CVaR_95 = 所有小于等于 VaR_95 的日收益平均值
```

若 VaR95 = -3%，直觉上表示样本中约 5% 的交易日亏损达到或超过 3%。若 CVaR95 = -5%，则最差 5% 日子的平均亏损约 5%。

**为什么 CVaR 很重要**  
VaR 只告诉你门槛，不告诉你越过门槛后有多惨；CVaR 补充尾部平均损失。

**如何应用**  
风险限额、资产比较、压力测试和仓位控制。

**图怎么看**  
在日收益直方图上，VaR 是左尾分位点，CVaR 更靠左，代表尾部平均。

**关联与配合**  
波动率描述整体，VaR/CVaR 描述左尾，最大回撤描述时间路径，三者互补。

**局限与误区**  
历史 VaR 假设未来类似历史；样本没有发生过的危机无法被估计。单日 VaR 也不能直接描述连续多日流动性危机。

In [ ]:
var_95 = df["ret_1d"].quantile(0.05)
cvar_95 = df.loc[df["ret_1d"] <= var_95, "ret_1d"].mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(df["ret_1d"].dropna(), bins=80, alpha=0.75)
ax.axvline(var_95, color="red", linestyle="--", label=f"VaR 95%={var_95:.2%}")
ax.axvline(cvar_95, color="purple", linestyle="--", label=f"CVaR 95%={cvar_95:.2%}")
ax.set_title("Daily Return Distribution")
ax.grid(True, alpha=0.3); ax.legend(); plt.show()
pd.Series({"VaR_95_daily": var_95, "CVaR_95_daily": cvar_95})

## 9. 相对市场指标：这只股票是在创造独立收益，还是只是跟着大盘？

单只股票上涨可能只是市场整体上涨。Beta、Alpha 和相关性把股票收益放到市场基准中比较。本例使用沪深 300；基准选择不同，结果也会改变。

### 9.1 Beta、Alpha、相关性

#### 为什么需要

如果大盘上涨 20%，一只股票上涨 15%，只看绝对收益可能不错，但相对市场并不强。我们需要拆分：

```text
股票超额收益 ≈ Alpha + Beta × 市场超额收益 + 残差
```

#### Beta

```text
Beta = Cov(StockExcess, MarketExcess) / Var(MarketExcess)
```

- Beta ≈ 1：和市场敏感度相近；
- Beta > 1：通常比市场波动更大；
- Beta < 1：市场敏感度较低；
- Beta < 0：与市场反向，但股票中较少见。

#### Alpha

Alpha 是带截距 CAPM 回归中的截距。本 Notebook 先减去日无风险收益，再用线性回归计算：

```text
StockExcess = Alpha_daily + Beta × MarketExcess + Residual
```

然后把日 Alpha 复利年化。正 Alpha 表示在该样本和该基准下，存在市场 Beta 未解释的平均超额收益。

#### 相关性

相关性只衡量同步方向，范围 -1 到 1；Beta 还受股票与市场各自波动率影响：

```text
Beta = Correlation × StockVol / MarketVol
```

#### 如何应用

- 组合风险暴露；
- 区分市场收益和个股收益；
- 选择对冲比例；
- 评价策略是否只是隐含做多市场。

#### 图怎么看

散点横轴是市场超额收益，纵轴是股票超额收益；回归线斜率是 Beta，截距是日 Alpha。

#### 局限与误区

Beta/Alpha 会随区间、频率和基准改变；Alpha 不是因果证明。单因子 CAPM 忽略行业、规模、价值、动量等风险因子。

In [ ]:
def load_hs300(start="20200101", end="20260101"):
    cache = DATA_DIR / f"hs300_{start}_{end}.csv"
    if cache.exists():
        return pd.read_csv(cache, index_col="Date", parse_dates=True)
    try:
        import akshare as ak
        raw = ak.stock_zh_index_daily(symbol="sh000300")
        idx = raw.rename(columns={"date": "Date", "open": "Open", "high": "High", "low": "Low", "close": "Close", "volume": "Volume"}).copy()
        idx["Date"] = pd.to_datetime(idx["Date"])
        idx = idx.set_index("Date").sort_index()
        idx = idx.loc[pd.to_datetime(start):pd.to_datetime(end)]
        idx = idx[["Open", "High", "Low", "Close", "Volume"]].dropna()
        idx.to_csv(cache, index_label="Date")
        return idx
    except Exception as e:
        print("沪深300下载失败，使用演示市场。错误：", repr(e))
        rng = np.random.default_rng(300)
        market_ret = df["ret_1d"].fillna(0) * 0.55 + rng.normal(0, 0.012, len(df))
        close = 4000 * (1 + pd.Series(market_ret, index=df.index)).cumprod()
        return pd.DataFrame({"Close": close}, index=df.index)

market = load_hs300()
combined = pd.concat({
    "stock": df["Close"].pct_change(),
    "market": market["Close"].pct_change(),
}, axis=1).dropna()

risk_free_annual = 0.02
risk_free_daily = (1 + risk_free_annual) ** (1 / 252) - 1
stock_excess = combined["stock"] - risk_free_daily
market_excess = combined["market"] - risk_free_daily

# 带截距的一元 CAPM 回归：slope=Beta，intercept=日 Alpha。
beta, alpha_daily = np.polyfit(market_excess, stock_excess, 1)
alpha_ann = (1 + alpha_daily) ** 252 - 1
corr = combined["stock"].corr(combined["market"])
residual = stock_excess - (alpha_daily + beta * market_excess)
assert abs(residual.mean()) < 1e-12

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(market_excess, stock_excess, alpha=0.35, s=12)
x = np.linspace(market_excess.min(), market_excess.max(), 100)
y = alpha_daily + beta * x
ax.plot(x, y, color="red", label=f"beta={beta:.2f}, annual alpha={alpha_ann:.2%}")
ax.set_title("CAPM: Stock Excess Return vs Market Excess Return")
ax.set_xlabel("Market daily excess return")
ax.set_ylabel("Stock daily excess return")
ax.grid(True, alpha=0.3); ax.legend(); plt.show()

pd.Series({
    "risk_free_annual": risk_free_annual,
    "beta": beta,
    "annualized_alpha": alpha_ann,
    "corr_with_market": corr,
})

## 10. 估值与基本面指标：公司赚不赚钱，市场为它付了多少

技术指标主要研究“价格如何变化”，基本面指标研究“公司经营如何”，估值指标研究“当前价格相对经营成果贵不贵”。财报是季度/年度低频数据，并存在公告滞后；回测时必须使用当时已经发布的数据，不能按报告期结束日提前看到财报。

### 10.1 PE / PB / PS / 股息率

#### 为什么需要

股价 5 元不代表便宜，500 元也不代表贵；还要看每股背后有多少利润、净资产、收入和分红。

#### PE 市盈率

```text
PE = MarketCap / NetProfit = Price / EPS
```

表示市场愿意为每 1 元利润支付多少价格。PE 20 可直觉理解为“按当前利润，价格约为年利润的 20 倍”，但它不是简单回本年限。

- 适合盈利相对稳定的公司；
- 净利润为负时 PE 失去直观意义；
- 周期股盈利高点时 PE 可能反而很低。

#### PB 市净率

```text
PB = MarketCap / BookEquity = Price / BookValuePerShare
```

表示为每 1 元账面净资产支付多少。常用于银行、保险、重资产行业，但账面资产质量差异很大。

#### PS 市销率

```text
PS = MarketCap / Revenue
```

适用于暂未盈利但有收入的公司；它忽略成本和利润率，低 PS 不代表能赚钱。

#### 股息率

```text
DividendYield = AnnualDividendPerShare / Price
```

表示按当前价格计算的现金分红收益率。高股息可能来自稳定分红，也可能只是股价大跌。

#### 如何应用

- 与自身历史分位比较；
- 与同行、同商业模式比较；
- 结合增长率、利润率、ROE 和资产负债率；
- 避免只看一个静态估值倍数。

#### 关联与相互作用

PE 与利润相关，PB 与净资产和 ROE 相关。理论上，在其他条件接近时，高 ROE、高增长公司可能拥有更高 PB/PE。PS 必须与利润率配合。

#### 局限与误区

TTM、静态、动态 PE 口径不同；一次性损益会扭曲利润。跨行业直接比较通常没有意义。

In [ ]:
def make_demo_valuation(index):
    rng = np.random.default_rng(42)
    val = pd.DataFrame(index=index)
    val["pe"] = 18 + np.sin(np.linspace(0, 10, len(index))) * 6 + rng.normal(0, 1.5, len(index))
    val["pb"] = 2.0 + np.sin(np.linspace(0, 8, len(index))) * 0.5 + rng.normal(0, 0.1, len(index))
    val["ps"] = 3.0 + np.sin(np.linspace(0, 6, len(index))) * 0.7 + rng.normal(0, 0.15, len(index))
    val["dv_ratio"] = 1.2 + rng.normal(0, 0.15, len(index))
    return val.clip(lower=0.01)


def load_valuation_603993():
    try:
        import akshare as ak
        raw = ak.stock_a_indicator_lg(symbol="603993")
        val = raw.copy()
        date_col = "trade_date" if "trade_date" in val.columns else val.columns[0]
        val[date_col] = pd.to_datetime(val[date_col])
        val = val.set_index(date_col).sort_index()
        cols = [c for c in ["pe", "pe_ttm", "pb", "ps", "ps_ttm", "dv_ratio", "total_mv"] if c in val.columns]
        for c in cols:
            val[c] = pd.to_numeric(val[c], errors="coerce")
        return val[cols].dropna(how="all")
    except Exception as e:
        print("估值数据下载失败，使用演示估值数据。错误：", repr(e))
        return make_demo_valuation(df.index)

valuation = load_valuation_603993()
plot_cols = [c for c in ["pe_ttm", "pe", "pb", "ps_ttm", "ps", "dv_ratio"] if c in valuation.columns]
fig, axes = plt.subplots(len(plot_cols), 1, figsize=(14, 3 * len(plot_cols)), sharex=True)
if len(plot_cols) == 1:
    axes = [axes]
for ax, col in zip(axes, plot_cols):
    ax.plot(valuation.index, valuation[col], label=col)
    ax.set_title(col)
    ax.grid(True, alpha=0.3)
    ax.legend()
plt.tight_layout(); plt.show()
valuation.tail()

### 10.2 ROE / ROA / 毛利率 / 净利率 / 资产负债率

#### 为什么需要

估值告诉我们市场付了多少钱，质量指标告诉我们公司用资产和股东资本赚得怎么样。

#### ROE 净资产收益率

```text
ROE = NetProfit / AverageShareholderEquity
```

表示每 1 元股东权益创造多少利润。严格计算应使用平均净资产；本例受财务摘要字段限制，以期末权益作教学近似。

高 ROE 可能来自：高利润率、高资产周转率，或高杠杆。杜邦分析把它拆成：

```text
ROE ≈ NetMargin × AssetTurnover × EquityMultiplier
```

#### ROA 总资产收益率

```text
ROA = NetProfit / AverageTotalAssets
```

衡量全部资产的赚钱效率，比 ROE 更少受资本结构影响。

#### 毛利率与净利率

```text
GrossMargin = (Revenue - OperatingCost) / Revenue
NetMargin = NetProfit / Revenue
```

毛利率看产品/业务层盈利空间，净利率看扣除费用、利息、税后最终留下多少。

#### 资产负债率

```text
DebtRatio = TotalLiabilities / TotalAssets
```

描述资产中有多少由负债支持。高负债不一定坏，需结合行业、现金流和利息保障倍数。

#### 如何应用

- 看多年趋势，而不是单期数字；
- 同行业比较；
- ROE 与负债率一起看，避免把杠杆误认为经营质量；
- 利润率和收入增长一起看，判断增长质量。

#### 图与数据说明

Notebook 优先读取 `603993` 的 AKShare 年报摘要。ROE、毛利率、净利率由原始财务值计算；ROA 和资产负债率使用摘要披露值。接口失败时会明确显示“演示数据”。

#### 局限与误区

财报低频且有公告延迟；会计政策、资产减值、一次性损益会影响指标。严谨回测必须按公告日对齐，避免未来函数。

In [ ]:
def make_demo_fundamentals():
    """仅在 AKShare 失败时使用；这些数字不是 603993 的真实财报。"""
    demo = pd.DataFrame({
        "Revenue": [100, 120, 150],
        "OperatingCost": [70, 78, 96],
        "NetProfit": [10, 14, 18],
        "Equity": [110, 125, 140],
        "ROA_reported": [0.05, 0.06, 0.07],
        "DebtRatio_reported": [0.45, 0.46, 0.47],
    }, index=pd.to_datetime(["2022-12-31", "2023-12-31", "2024-12-31"]))
    demo.attrs["source"] = "演示数据（不是 603993 实际财报）"
    return demo


def load_fundamentals_603993(start_year=2020):
    try:
        import akshare as ak
        raw = ak.stock_financial_abstract(symbol="603993")

        def metric(name):
            rows = raw[(raw["选项"] == "常用指标") & (raw["指标"] == name)]
            if len(rows) != 1:
                raise ValueError(f"无法唯一定位财务指标：{name}")
            return pd.to_numeric(rows.iloc[0, 2:], errors="coerce")

        date_cols = pd.Index(raw.columns[2:].astype(str))
        annual_cols = [c for c in date_cols if c.endswith("1231") and int(c[:4]) >= start_year]
        data = pd.DataFrame(index=pd.to_datetime(annual_cols))
        data["Revenue"] = metric("营业总收入").reindex(annual_cols).to_numpy()
        data["OperatingCost"] = metric("营业成本").reindex(annual_cols).to_numpy()
        data["NetProfit"] = metric("归母净利润").reindex(annual_cols).to_numpy()
        data["Equity"] = metric("股东权益合计(净资产)").reindex(annual_cols).to_numpy()
        data["ROA_reported"] = metric("总资产报酬率(ROA)").reindex(annual_cols).to_numpy() / 100
        data["DebtRatio_reported"] = metric("资产负债率").reindex(annual_cols).to_numpy() / 100
        data = data.sort_index().dropna(subset=["Revenue", "NetProfit"])
        data.attrs["source"] = "AKShare stock_financial_abstract：603993 年报"
        return data
    except Exception as e:
        print("603993 财务摘要下载失败，使用明确标记的演示数据。错误：", repr(e))
        return make_demo_fundamentals()

fundamentals = load_fundamentals_603993()
fundamentals["ROE"] = fundamentals["NetProfit"] / fundamentals["Equity"]
fundamentals["ROA"] = fundamentals["ROA_reported"]
fundamentals["GrossMargin"] = (fundamentals["Revenue"] - fundamentals["OperatingCost"]) / fundamentals["Revenue"]
fundamentals["NetMargin"] = fundamentals["NetProfit"] / fundamentals["Revenue"]
fundamentals["DebtRatio"] = fundamentals["DebtRatio_reported"]

source_label = fundamentals.attrs.get("source", "未知来源")
print("数据来源：", source_label)
plot_lines(
    fundamentals,
    ["ROE", "ROA", "GrossMargin", "NetMargin", "DebtRatio"],
    f"603993 Fundamental Ratios — {source_label}",
)
fundamentals

## 11. 指标之间怎么配合：每个指标负责一个问题

不要把十个相似指标当作十张赞成票。SMA、EMA、MACD、BBI 都大量依赖同一份价格趋势信息；RSI、KDJ、Williams %R 也高度相关。重复指标只会制造虚假的“多重确认”。

更合理的组合方式是每一类只选一个代表：

```text
环境：市场是趋势还是震荡？       → ADX
方向：趋势向上还是向下？         → SMA/EMA/MACD 中选一类
风险：当前波动是否可接受？       → Volatility/ATR
位置：是否过度偏离？             → RSI/BIAS/%B 中选一个
确认：成交参与是否足够？         → VolumeRatio/CMF
可交易性：资金是否真的进得去？   → Amount/Amihud
估值与质量：贵不贵、好不好？     → PE/PB + ROE/Margin/Debt
```

### 一个教学组合

```text
1. Close > SMA60：趋势向上
2. ADX > 20 且 +DI > -DI：趋势有一定强度且方向向上
3. Vol20 低于风险上限：波动可接受
4. VolumeRatio20 > 1：突破有成交量参与
5. RSI < 80：避免极端追高
```

每增加一个条件，交易次数会减少、过拟合风险会上升。因此必须检查：

- 每个条件是否有不同经济含义；
- 去掉某个条件后结果是否崩溃；
- 参数附近是否稳定；
- 样本外、不同股票和不同市场环境是否仍有效；
- 手续费、滑点、涨跌停和流动性后是否还能成交。

## 12. 逐项计算自检：公式是否满足基本性质？

“代码能运行”不等于“指标算对了”。下面不拿另一个指标库当黑盒，而是检查每类指标应满足的基本数学性质：

- 累计收益应等于首尾价格比；
- RSI/MFI/ADX 应大致落在 0–100；
- Williams %R 应落在 -100–0；
- 上轨应不低于中轨，中轨应不低于下轨；
- ATR、波动率、成交量比和非流动性不能为负；
- 回撤不能为正；
- CVaR 应不高于 VaR；
- MACD 柱必须等于 DIF−DEA。

这些检查只能发现明显的公式和实现错误，不能证明指标具有预测能力。

In [ ]:
def finite_between(series, low, high):
    values = series.dropna()
    return not values.empty and values.between(low, high).all()

checks = {
    "累计收益等于首尾价格复利收益": np.isclose(
        1 + df["cum_ret"].dropna().iloc[-1],
        df["Close"].iloc[-1] / df["Close"].iloc[0],
    ),
    "回撤不大于 0": (df["drawdown"].dropna() <= 1e-12).all(),
    "SMA20 最后值等于最后 20 日均值": np.isclose(
        df["sma_20"].dropna().iloc[-1], df["Close"].iloc[-20:].mean()
    ),
    "BIAS20 符合定义": np.isclose(
        df["bias_20"].dropna().iloc[-1],
        df["Close"].iloc[-1] / df["sma_20"].iloc[-1] - 1,
    ),
    "MACD 柱等于 DIF-DEA": np.allclose(
        df["macd_hist"].dropna(),
        (df["macd_dif"] - df["macd_dea"]).dropna(),
    ),
    "RSI14 位于 0-100": finite_between(df["rsi_14"], 0, 100),
    "K/D 位于 0-100": finite_between(df["kdj_k"], 0, 100) and finite_between(df["kdj_d"], 0, 100),
    "Williams %R 位于 -100-0": finite_between(df["willr_14"], -100, 0),
    "历史波动率非负": (df["vol_20"].dropna() >= 0).all(),
    "布林带上中下轨顺序正确": (
        (df["bb_upper"].dropna() >= df.loc[df["bb_upper"].dropna().index, "bb_mid"]).all()
        and (df["bb_mid"].dropna() >= df.loc[df["bb_mid"].dropna().index, "bb_lower"]).all()
    ),
    "ATR/NATR 非负": (df[["atr_14", "natr_14"]].dropna() >= 0).all().all(),
    "Donchian 上中下轨顺序正确": (
        (df["donchian_upper_20"].dropna() >= df.loc[df["donchian_upper_20"].dropna().index, "donchian_mid_20"]).all()
        and (df["donchian_mid_20"].dropna() >= df.loc[df["donchian_mid_20"].dropna().index, "donchian_lower_20"]).all()
    ),
    "Keltner 上中下轨顺序正确": (lambda bands: (
        (bands["kc_upper"] >= bands["kc_mid"]).all()
        and (bands["kc_mid"] >= bands["kc_lower"]).all()
    ))(df[["kc_upper", "kc_mid", "kc_lower"]].dropna()),
    "ADX 位于 0-100": finite_between(df["adx_14"], 0, 100),
    "成交量比非负": (df["volume_ratio_20"].dropna() >= 0).all(),
    "CMF 位于 -1-1": finite_between(df["cmf_20"], -1, 1),
    "MFI 位于 0-100": finite_between(df["mfi_14"], 0, 100),
    "滚动 VWAP 为正": (df["vwap_20"].dropna() > 0).all(),
    "VR 非负": (df["vr_26"].dropna() >= 0).all(),
    "Amihud 非负": (df["amihud"].dropna() >= 0).all(),
    "CVaR 不高于 VaR": cvar_95 <= var_95,
    "Beta/Alpha/相关性为有限数": np.isfinite([beta, alpha_ann, corr]).all(),
    "基本面比率为有限数": np.isfinite(
        fundamentals[["ROE", "ROA", "GrossMargin", "NetMargin", "DebtRatio"]].dropna().to_numpy()
    ).all(),
}

validation = pd.Series(checks, name="通过")
display(validation.to_frame())
assert validation.all(), validation[~validation]
print(f"全部 {len(validation)} 项基础计算检查通过。")

## 13. 常见技术指标覆盖清单

本 Notebook 已覆盖常见技术指标的主要类别：

| 类别 | 指标 |
|---|---|
| 价格收益 | 简单收益、对数收益、累计收益、回撤 |
| 趋势 | SMA、EMA、BIAS、MACD、BBI |
| 动量/震荡 | ROC、RSI、KDJ、CCI、Williams %R |
| 波动/通道 | 历史波动率、Bollinger Bands、ATR、NATR、Donchian、Keltner |
| 趋势强度/止损 | DMI、ADX、Parabolic SAR |
| 成交量/资金流 | Volume Ratio、OBV、PVT、ADL、CMF、MFI、VWAP、VR |
| 流动性 | Amihud |
| 风险绩效 | 年化收益、年化波动、夏普、滚动夏普、VaR、CVaR |
| 相对市场 | Beta、Alpha、相关性 |
| 估值/基本面 | PE、PB、PS、股息率、ROE、ROA、毛利率、净利率、资产负债率 |

没有追求 100% 覆盖冷门指标。很多冷门指标只是这些指标的变体。

## 14. 导出 HTML

```bash
uv run jupyter nbconvert --to html notebooks/金融股票指标入门_matplotlib.ipynb
```

执行全部代码后导出：

```bash
uv run jupyter nbconvert \
  --to html \
  --execute \
  --ExecutePreprocessor.timeout=600 \
  notebooks/金融股票指标入门_matplotlib.ipynb
```